<a href="https://colab.research.google.com/github/Dilshan-Prince/Statistical-Learning-e21100/blob/main/data_anylysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part A: Theoretical Fundamentals (Maximum Likelihood & Decision Space)1. Maximum Likelihood Bias and Bessel's CorrectionDeriving the Bias of $\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}$:The raw Maximum Likelihood Estimator for the covariance matrix is:$$\widehat{\boldsymbol{\Sigma}}_{\text{MLE}} = \frac{1}{n}\sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T$$By expanding the quadratic term inside the sum:$$\sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T = \sum_{i=1}^n \mathbf{X}_i\mathbf{X}_i^T - n\widehat{\boldsymbol{\mu}}_n\widehat{\boldsymbol{\mu}}_n^T$$Taking the expectation of both sides requires us to evaluate the second moments. Using the standard variance identity $\mathrm{Var}(\mathbf{Y}) = \mathbb{E}[\mathbf{Y}\mathbf{Y}^T] - \mathbb{E}[\mathbf{Y}]\mathbb{E}[\mathbf{Y}]^T$:For the individual observations: $\mathbb{E}[\mathbf{X}_i\mathbf{X}_i^T] = \boldsymbol{\Sigma} + \boldsymbol{\mu}\boldsymbol{\mu}^T$For the sample mean: $\mathbb{E}[\widehat{\boldsymbol{\mu}}_n\widehat{\boldsymbol{\mu}}_n^T] = \frac{1}{n}\boldsymbol{\Sigma} + \boldsymbol{\mu}\boldsymbol{\mu}^T$Substituting these into the expectation operator:$$\mathbb{E}\left[\sum_{i=1}^n (\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)(\mathbf{X}_i - \widehat{\boldsymbol{\mu}}_n)^T\right] = n(\boldsymbol{\Sigma} + \boldsymbol{\mu}\boldsymbol{\mu}^T) - n\left(\frac{1}{n}\boldsymbol{\Sigma} + \boldsymbol{\mu}\boldsymbol{\mu}^T\right) = (n - 1)\boldsymbol{\Sigma}$$Dividing by $n$ completes the proof of bias:$$\mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \frac{n-1}{n}\boldsymbol{\Sigma}$$Bessel's Correction:To correct this systemic underestimation, we multiply the MLE estimator by the factor $\frac{n}{n-1}$. This defines the unbiased sample covariance matrix $\mathbf{S}$:$$\mathbb{E}[\mathbf{S}] = \mathbb{E}\left[\frac{n}{n-1}\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}\right] = \frac{n}{n-1} \left( \frac{n-1}{n}\boldsymbol{\Sigma} \right) = \boldsymbol{\Sigma}$$


2. SHM Errors and the Decision SpaceType I Error ($\alpha$): A false alarm. This occurs when the diagnostic layer incorrectly concludes that the structure is damaged or shifting, when in reality, it remains perfectly healthy.Type II Error ($\beta$): A missed detection. This occurs when the diagnostic layer incorrectly concludes that the structure is healthy, failing to flag a true internal fracture or anomaly.Consequence of Ultra-Conservative Threshold ($\alpha = 0.0001$): By drastically lowering the false-alarm threshold, the engineer forces the system to require overwhelming evidence to trigger an alarm. Geometrically, this massively expands the volume of the "healthy operation" confidence ellipsoid in the feature space. Mathematically, minimizing $\alpha$ intrinsically inflates the Type II error rate ($\beta$), which severely degrades the statistical power ($1-\beta$) of the diagnostic layer. The system becomes highly insensitive to subtle structural degradations.Part B: Theoretical Extension (Asymptotic Distributions & Slutsky's Theorem)1. Slutsky's TheoremSlutsky's Theorem states that if a sequence of random vectors $\mathbf{X}_n$ converges in distribution to a random vector $\mathbf{X}$ ($\mathbf{X}_n \xrightarrow{d} \mathbf{X}$), and a sequence of random matrices $\mathbf{Y}_n$ converges in probability to a constant matrix $\mathbf{C}$ ($\mathbf{Y}_n \xrightarrow{p} \mathbf{C}$), then:$$\mathbf{Y}_n \mathbf{X}_n \xrightarrow{d} \mathbf{C} \mathbf{X}$$2. Justifying the Substitution of $\mathbf{S}$By the Weak Law of Large Numbers, the unbiased sample covariance matrix is a consistent estimator of the population covariance, meaning $\mathbf{S} \xrightarrow{p} \boldsymbol{\Sigma}$ as $n \to \infty$. By the continuous mapping theorem, this implies $\mathbf{S}^{-1/2} \xrightarrow{p} \boldsymbol{\Sigma}^{-1/2}$.The Lindeberg-Lévy CLT gives us:$$\sqrt{n}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \mathscr{N}(\mathbf{0}, \boldsymbol{\Sigma})$$Pre-multiplying by $\mathbf{S}^{-1/2}$ and applying Slutsky's Theorem allows us to swap the latent parameter for the empirical estimator without breaking the asymptotic Gaussian convergence:$$\sqrt{n}\mathbf{S}^{-1/2}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \mathscr{N}(\mathbf{0}, \mathbf{I})$$By reversing the standardization algebraically, we operationalize the practical parametric modeling distribution:$$\widehat{\boldsymbol{\mu}}_n \sim \mathscr{N}\left({\boldsymbol{\mu}}, \frac{1}{n}\mathbf{S}\right)$$

In [ ]:
#Part C Numerical Verification
import numpy as np
import pandas as pd
import scipy.stats as stats

# --- DO NOT MODIFY: Synthetic Strain Sensor Data Generator ---
np.random.seed(42)
n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

base_data[4000:, 0] += 0.015  # Sensor 1 drift
base_data[4000:, 2] -= 0.010  # Sensor 3 drift

df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])
# ----------------------------------------------------------------

def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    """
    Partitions the dataset into g chunks and evaluates first-moment homogeneity
    via Wilks' Lambda and Bartlett's Chi-Square asymptotic transformation.
    """
    data = df.values
    n, m = data.shape
    n_j = n // g_chunks

    # 1. Global Mean
    global_mean = np.mean(data, axis=0)

    # 2. Initialize Within (W) and Total (T) scatter matrices
    W = np.zeros((m, m))
    T = np.zeros((m, m))

    # Calculate Total Scatter Matrix (T)
    for i in range(n):
        diff = (data[i] - global_mean).reshape(-1, 1)
        T += diff @ diff.T

    # Calculate Within-Chunk Scatter Matrix (W)
    for j in range(g_chunks):
        chunk = data[j*n_j : (j+1)*n_j]
        chunk_mean = np.mean(chunk, axis=0)
        for i in range(n_j):
            diff = (chunk[i] - chunk_mean).reshape(-1, 1)
            W += diff @ diff.T

    # Calculate Between-Chunk Scatter Matrix (B)
    B = T - W

    # 3. Compute Wilks' Lambda
    det_W = np.linalg.det(W)
    det_T = np.linalg.det(T)
    wilks_lambda = det_W / det_T

    # 4. Bartlett's Chi-Square Approximation
    chi2_stat = -(n - 1 - (m + g_chunks) / 2) * np.log(wilks_lambda)
    df_chi2 = m * (g_chunks - 1)
    p_value = 1.0 - stats.chi2.cdf(chi2_stat, df_chi2)

    conclusion = "Baseline Center Shift Detected (Reject H0)" if p_value < 0.05 else "First Moment Homogeneous (Fail to Reject H0)"

    return {
        "Wilks' Lambda": np.round(wilks_lambda, 6),
        "Bartlett Chi-Square": np.round(chi2_stat, 2),
        "Degrees of Freedom": df_chi2,
        "P-Value": p_value,
        "Conclusion": conclusion
    }

# Execute
results = verify_first_moment_homogeneity(df_strain, g_chunks=5)
for key, value in results.items():
    print(f"{key}: {value}")

Geometric Subspace Optimization via Principal Component Analysis (PCA)Part A: Theoretical Fundamentals1. Coordinate Projections & Orthogonality$$\mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T] = \mathbb{E}[(\mathbf{P}^T \widetilde{\mathbf{X}}_i)(\mathbf{P}^T \widetilde{\mathbf{X}}_i)^T] = \mathbb{E}[\mathbf{P}^T \widetilde{\mathbf{X}}_i \widetilde{\mathbf{X}}_i^T \mathbf{P}]$$Because $\mathbf{P}$ is a matrix of constants (the deterministic eigenvectors), we pull it outside the expectation operator:$$= \mathbf{P}^T \mathbb{E}[\widetilde{\mathbf{X}}_i \widetilde{\mathbf{X}}_i^T] \mathbf{P} = \mathbf{P}^T \boldsymbol{\Sigma} \mathbf{P}$$Substituting the spectral decomposition $\boldsymbol{\Sigma} = \mathbf{P} \mathbf{\Lambda} \mathbf{P}^T$ and recalling that $\mathbf{P}^T \mathbf{P} = \mathbf{I}$ (due to orthogonality):$$= \mathbf{P}^T (\mathbf{P} \mathbf{\Lambda} \mathbf{P}^T) \mathbf{P} = (\mathbf{P}^T \mathbf{P}) \mathbf{\Lambda} (\mathbf{P}^T \mathbf{P}) = \mathbf{I} \mathbf{\Lambda} \mathbf{I} = \mathbf{\Lambda}$$Geometric Meaning: Because $\mathbf{\Lambda}$ is strictly diagonal, the cross-covariance between any distinct coordinates $Z_{i,j}$ and $Z_{i,k}$ is identically zero. Geometrically, the data has been projected onto a new set of strictly orthogonal axes where all linear correlations have been entirely eliminated.2. Total Variance ConservationApplying the cyclic permutation property of the trace operator ($\text{tr}(ABC) = \text{tr}(BCA)$):$$\text{tr}(\boldsymbol{\Sigma}) = \text{tr}(\mathbf{P} \mathbf{\Lambda} \mathbf{P}^T) = \text{tr}(\mathbf{\Lambda} \mathbf{P}^T \mathbf{P}) = \text{tr}(\mathbf{\Lambda} \mathbf{I}) = \text{tr}(\mathbf{\Lambda}) = \sum_{j=1}^m \lambda_j$$Cumulative Explained Variance Ratio: $\Phi(k) = \frac{\sum_{j=1}^k \lambda_j}{\sum_{j=1}^m \lambda_j}$Residual Unexplained Variance Ratio: $\Psi(k) = \frac{\sum_{j=k+1}^m \lambda_j}{\sum_{j=1}^m \lambda_j} = 1 - \Phi(k)$3. Vector Decomposition and Subspace StatisticsReconstructed Vector: $\widehat{\mathbf{x}}_i = \widehat{\mathbf{P}}_k \mathbf{z}_{i,k}$Residual Error Vector: $\mathbf{e}_i = \widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}$Because the principal components form an orthonormal basis, the subspace spanned by $\widehat{\mathbf{P}}_k$ is strictly orthogonal to the subspace spanned by $\widehat{\mathbf{P}}_{m-k}$. Thus, the dot product of their vectors is zero, allowing the Pythagorean Theorem to apply perfectly in high-dimensional space:$$\|\mathbf{x}_i - \widehat{\boldsymbol{\mu}}_n\|^2 = \|\widehat{\mathbf{x}}_i + \mathbf{e}_i\|^2 = \|\widehat{\mathbf{x}}_i\|^2 + \|\mathbf{e}_i\|^2 = \|\mathbf{z}_{i,k}\|^2 + \|\mathbf{z}_{i,m-k}\|^2$$Contrast of Statistics ($T^2$ vs $Q$):Hotelling's $T^2$: Monitors variations within the explained principal subspace. It is most sensitive to environmental load anomalies (e.g., extreme wind). The structure's physical correlation matrix remains intact, but the magnitude of the normal loading simply scales up along the expected axes.$Q$ Statistic (SPE): Monitors variations in the residual subspace. It is most sensitive to internal structural fractures. A fracture physically alters the load pathways, breaking the established correlation geometry between sensors and forcing energy out of the principal plane and into the residual error space.

In [ ]:
#Part B : PCA code for numerical verification
import numpy as np

class PCA_Diagnostics:
    def __init__(self, data: np.ndarray):
        self.data = data
        self.n, self.m = data.shape

    def evaluate_truncation_boundaries(self):
        # 1. Standardize and center
        mean = np.mean(self.data, axis=0)
        std = np.std(self.data, axis=0, ddof=1)
        Z = (self.data - mean) / std

        # 2. Covariance and spectral decomposition
        S = np.cov(Z, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(S)

        # Sort in descending order
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        # Project into full Z-space
        Z_scores = Z @ eigenvectors

        results = {}
        # 3 & 4. Loop through cutoff values k
        for k in [1, 2, 3]:
            # Hotelling's T^2 calculation
            T2_array = np.sum((Z_scores[:, :k]**2) / eigenvalues[:k], axis=1)
            mean_T2 = np.mean(T2_array)

            # Q Statistic (SPE) calculation
            Q_array = np.sum((Z_scores[:, k:]**2), axis=1)
            mean_Q = np.mean(Q_array)

            results[f"k={k}"] = {"Mean T^2": np.round(mean_T2, 3), "Mean Q": np.round(mean_Q, 3)}

        return results

# Example Execution (using the strain data from earlier)
# pca_engine = PCA_Diagnostics(df_strain.values)
# print(pca_engine.evaluate_truncation_boundaries())

Latent Subspace Decomposition via Factor Analysis (FA)Part A: Theoretical Fundamentals1. Fundamental Equation and Communalities$$\mathbf{R} = \mathbb{E}[(\boldsymbol{\Lambda}\mathbf{F}_i + \boldsymbol{\epsilon}_i)(\boldsymbol{\Lambda}\mathbf{F}_i + \boldsymbol{\epsilon}_i)^T]$$Expanding the quadratic and applying the expectation operator linearly:$$= \boldsymbol{\Lambda}\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T]\boldsymbol{\Lambda}^T + \boldsymbol{\Lambda}\mathbb{E}[\mathbf{F}_i\boldsymbol{\epsilon}_i^T] + \mathbb{E}[\boldsymbol{\epsilon}_i\mathbf{F}_i^T]\boldsymbol{\Lambda}^T + \mathbb{E}[\boldsymbol{\epsilon}_i\boldsymbol{\epsilon}_i^T]$$Given that the factors are orthogonal ($\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T] = \mathbf{I}$), and independent of the errors ($\mathbb{E}[\mathbf{F}_i\boldsymbol{\epsilon}_i^T] = \mathbf{0}$):$$\mathbf{R} = \boldsymbol{\Lambda}\mathbf{I}\boldsymbol{\Lambda}^T + \mathbf{0} + \mathbf{0} + \boldsymbol{\Psi} = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$$Communality ($h_j^2$): The sum of squared factor loadings for sensor $j$. It represents the proportion of that specific sensor's variance that is driven by the global shared physical mechanics of the structure.Uniqueness ($\varphi_j^2$): The diagonal entry of $\boldsymbol{\Psi}$. It represents variance strictly isolated to sensor $j$, quantifying specific localized instrument noise or a localized electrical fault.2. Varimax RotationRaw PCA vs. Rotated FA: Raw PCA forces eigenvectors into a strict mathematical hierarchy ($\lambda_1 > \lambda_2 \dots$), meaning the first component sweeps up mixed variance across almost all sensors. Varimax rotation abandons this hierarchy, rotating the axes within the subspace to polarize the loadings toward 1 or 0. This creates "simple structure," mapping isolated groups of physical sensors to specific factors.Mathematical Invariance: If $\mathbf{T}$ is an orthogonal rotation matrix ($\mathbf{T}\mathbf{T}^T = \mathbf{I}$), applying it yields $\boldsymbol{\Lambda}_{\text{rot}} = \boldsymbol{\Lambda}\mathbf{T}$. Substituting this back into the generative model yields $\boldsymbol{\Lambda}_{\text{rot}}\boldsymbol{\Lambda}_{\text{rot}}^T = \boldsymbol{\Lambda}\mathbf{T}\mathbf{T}^T\boldsymbol{\Lambda}^T = \boldsymbol{\Lambda}\mathbf{I}\boldsymbol{\Lambda}^T = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T$. The global covariance approximation and individual row sums of squares ($h_j^2$) are completely invariant to the rotation.3. Thomson’s Regression MethodThe off-diagonal block $\boldsymbol{\Sigma}_{ZF} = \mathbb{E}[\mathbf{Z}_i \mathbf{F}_i^T] = \mathbb{E}[(\boldsymbol{\Lambda}\mathbf{F}_i + \boldsymbol{\epsilon}_i)\mathbf{F}_i^T] = \boldsymbol{\Lambda}\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T] + \mathbb{E}[\boldsymbol{\epsilon}_i\mathbf{F}_i^T] = \boldsymbol{\Lambda}$. Therefore, the joint covariance expands to:$$\boldsymbol{\Sigma}_{YY} = \begin{bmatrix} \mathbb{E}[\mathbf{Z}_i\mathbf{Z}_i^T] & \mathbb{E}[\mathbf{Z}_i\mathbf{F}_i^T] \\ \mathbb{E}[\mathbf{F}_i\mathbf{Z}_i^T] & \mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T] \end{bmatrix} = \begin{bmatrix} \mathbf{R} & \boldsymbol{\Lambda} \\ \boldsymbol{\Lambda}^T & \mathbf{I}_{k \times k} \end{bmatrix}$$Applying the conditional expectation identity where $\boldsymbol{\mu}_2 = \mathbf{0}$, $\boldsymbol{\mu}_1 = \mathbf{0}$, $\boldsymbol{\Sigma}_{21} = \boldsymbol{\Lambda}^T$, and $\boldsymbol{\Sigma}_{11}^{-1} = \mathbf{R}^{-1}$:$$\mathbf{f}_i = \mathbb{E}[\mathbf{F}_i | \mathbf{z}_i] = \mathbf{0} + \boldsymbol{\Lambda}^T \mathbf{R}^{-1} (\mathbf{z}_i - \mathbf{0}) = \boldsymbol{\Lambda}^T \mathbf{R}^{-1} \mathbf{z}_i$$

In [ ]:
#Parametric Factor Partition Engine & UI Architecture
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import FactorAnalysis

def execute_diagnostic_fa_pipeline(df: pd.DataFrame, k: int):
    # 1. Z-Score Mapping Layer with Guardrails
    raw_data = df.select_dtypes(include=[np.number]).values
    n, m = raw_data.shape

    std_dev = np.std(raw_data, axis=0, ddof=1)
    std_dev[std_dev == 0] = 1e-15 # Zero-variance limit boundary

    Z = (raw_data - np.mean(raw_data, axis=0)) / std_dev

    # 2. Dimensionality Constraints
    if n <= m:
        raise ValueError(f"Operational snapshot count ({n}) must exceed channel count ({m}).")
    if k >= m:
        raise ValueError(f"Latent space target ({k}) must be strictly less than feature dimension ({m}).")

    # 3. Information Decomposition Layer (SVD + Varimax)
    fa = FactorAnalysis(n_components=k, rotation='varimax', random_state=42)
    fa.fit(Z)

    # Extract components (Loadings matrix Λ transposed in sklearn)
    Lambda_rot = fa.components_.T

    # Communality and Uniqueness
    communality = np.sum(Lambda_rot**2, axis=1)
    uniqueness = fa.noise_variance_

    # 4. Mathematical State Reconstruction (Thomson's Method)
    R = np.corrcoef(Z, rowvar=False)
    R_inv = np.linalg.inv(R)
    # Thomson's formula: F = Z * R^-1 * Lambda
    F_scores = Z @ R_inv @ Lambda_rot

    # 5. Diagnostic Telemetry Logging
    emp_variance = np.var(F_scores, axis=0, ddof=1)
    avg_comm = np.mean(communality) * 100
    avg_uniq = np.mean(uniqueness) * 100

    print("--- SYSTEM DIAGNOSTIC TELEMETRY ---")
    print(f"Average System Communality : {avg_comm:.2f}%")
    print(f"Average System Uniqueness  : {avg_uniq:.2f}%")

    # 6. UI Architecture Rendering
    fig = make_subplots(
        rows=2, cols=2,
        horizontal_spacing=0.24, vertical_spacing=0.28,
        subplot_titles=(
            "Structural Loadings |λ_(j,r)|",
            "Variance Partitioning Profile",
            "Sensor Uniqueness Noise Floor",
            "Latent Empirical Variance"
        )
    )

    sensor_labels = df.columns
    factor_labels = [f"Factor {i+1}" for i in range(k)]

    # [1,1] Heatmap
    fig.add_trace(
        go.Heatmap(
            z=np.abs(Lambda_rot), x=factor_labels, y=sensor_labels,
            colorscale='YlOrRd', showscale=True,
            colorbar=dict(x=-0.15, len=0.38, y=0.78, yanchor="middle", xanchor="right")
        ), row=1, col=1
    )

    # [1,2] Variance Partitioning (Horizontal Stacked Bar)
    fig.add_trace(go.Bar(y=sensor_labels, x=communality*100, name='Communality (h²)', orientation='h', marker_color='#1f77b4'), row=1, col=2)
    fig.add_trace(go.Bar(y=sensor_labels, x=uniqueness*100, name='Uniqueness (φ²)', orientation='h', marker_color='#ff7f0e'), row=1, col=2)
    fig.update_xaxes(range=[0, 100], title_text="Percentage (%)", row=1, col=2)

    # [2,1] Sensor Uniqueness Line Profile
    fig.add_trace(
        go.Scatter(
            x=sensor_labels, y=uniqueness, name='Uniqueness',
            mode='lines+markers', line=dict(color='#d62728', width=2, dash='dashdot'),
            marker=dict(symbol='x', size=8)
        ), row=2, col=1
    )
    fig.update_xaxes(tickangle=25, row=2, col=1)

    # [2,2] Latent Factor Variance Distribution
    fig.add_trace(
        go.Bar(
            x=factor_labels, y=emp_variance, name='Score Variance',
            marker=dict(color='#2ca02c', line=dict(color='black', width=0.5))
        ), row=2, col=2
    )

    # Global Layout constraints
    fig.update_layout(
        width=1250, height=750, template='plotly_white', barmode='stack',
        legend=dict(orientation="h", x=0.5, y=1.02, xanchor="center", yanchor="bottom"),
        margin=dict(t=150, b=60, l=140, r=80)
    )

    fig.show()

# Execution syntax:
# execute_diagnostic_fa_pipeline(df_asset, k=2)

Part B: Analysis Questions Question

 1: The Total Variance IllusionA Uniqueness value ($\varphi^2$) approaching $100\%$ explicitly confirms that Sensor_4's data is disconnected from the global structural mechanics. It is dominated entirely by localized electrical noise or a broken instrumentation loop, providing zero insight into the actual physical asset.Because PCA is a geometric algorithm designed to maximize raw total variance indiscriminately, it cannot distinguish between meaningful structural variance and high-magnitude sensor noise. Because Sensor_4 has a massive internal variance scale, PCA will inherently construct a dominant principal component just to accommodate this localized noise, confusing volume with structural importance.If an automated plant anomaly detection framework relies purely on PCA, the boundaries for Hotelling's $T^2$ will be heavily contaminated by Sensor_4. The diagnostic layer will trigger false alarms based on random electrical noise spikes rather than true mechanical failures.Question 2: Decoupling Structural Loading via RotationTraditional PCA forces the first component to sweep up as much mixed variance as possible across all sensors, creating highly intertwined eigenvector weights. By abandoning this strict variance hierarchy, the Varimax rotation iteratively rotates the axes to polarize the loading matrix. It forces weights toward exactly 1 or 0, yielding "simple structure" where sensors are explicitly mapped to distinct, uncoupled latent physical groups.A rotated FA heatmap is vastly easier for a plant operator to troubleshoot. If "Factor 1" indicates an anomaly, the operator immediately knows it corresponds exactly to the physical zone monitored by Sensor_1 and Sensor_2. Unrotated PCA matrices mix all sensors together, making it nearly impossible to isolate exactly which physical subsystem is degrading without extensive manual analysis.Question 3: Determining Subspace Truncation ($k$)The Mean $Q$ Statistic drops sharply and steeply when moving from $k=1$ to $k=2$. However, when transitioning from $k=2$ to $k=3$, the curve flattens out, forming a distinct horizontal plateau (an "elbow").The sharp initial drop proves that meaningful structural variance is successfully being transferred into the tracked $T^2$ subspace. The flat elbow at $k=2$ indicates that adding a third dimension captures absolutely no new structural mechanics; it only absorbs random, isolated white noise. Thus, $k=2$ is the true hidden physical dimensionality.If you incorrectly set $k=3$, you are aggressively overfitting the model. You are forcing meaningless white noise and local sensor instrumentation errors directly into the "clean" principal tracking subspace, effectively corrupting your baseline.Question 4: Operational Trade-offsThe PCA Strategy: Highly susceptible to localized noise inflation. A single malfunctioning sensor will warp the principal geometry and bleed into the $T^2$ boundary, disrupting the entire system.The FA Strategy: The generative model is mathematically designed to isolate independent local faults into the diagonal Uniqueness matrix ($\boldsymbol{\Psi}$), keeping the shared structural subspace perfectly clean.Conclusion: The FA Strategy is significantly more robust. If a specific instrument loses calibration or shorts out (like Sensor_4), the shared Latent Factor scores remain untainted, protecting the structural diagnosis. Meanwhile, looking directly at the Sensor Uniqueness Noise Floor ($\varphi^2$) will instantly flag Sensor_4 as defective, allowing maintenance teams to easily distinguish an isolated instrument failure from a catastrophic structural load shift.